In [ ]:
import os
from pathlib import Path
from PIL import Image

INPUT_LABEL_FILE = "labels/train/wider_face_train_bbx_gt.txt"
INPUT_IMAGE_DIR = "images/train"
OUTPUT_IMAGE_DIR = "widerface_yolo/images/train"
OUTPUT_LABEL_DIR = "widerface_yolo/labels/train"

os.makedirs(OUTPUT_IMAGE_DIR, exist_ok=True)
os.makedirs(OUTPUT_LABEL_DIR, exist_ok=True)

with open(INPUT_LABEL_FILE, "r") as f:
    lines = f.readlines()

i = 0
while i < len(lines):
    image_path_rel = lines[i].strip()
    i += 1
    try :
        face_count = int(lines[i].strip())
        i += 1
    except:
        print(face_count)

    full_image_path = os.path.join(INPUT_IMAGE_DIR, image_path_rel)
    output_image_path = os.path.join(OUTPUT_IMAGE_DIR, os.path.basename(image_path_rel))
    output_label_path = os.path.join(OUTPUT_LABEL_DIR, Path(image_path_rel).with_suffix(".txt").name)

    # Copier l'image
    os.makedirs(os.path.dirname(output_image_path), exist_ok=True)
    os.system(f'cp "{full_image_path}" "{output_image_path}"')

    # Lire taille image
    with Image.open(full_image_path) as img:
        W, H = img.size

    yolo_labels = []
    for _ in range(face_count):
        parts = list(map(int, lines[i].strip().split()))
        i += 1
        x, y, w, h = parts[0:4]

        # Conversion YOLO (normalisée)
        x_center = (x + w / 2) / W
        y_center = (y + h / 2) / H
        width = w / W
        height = h / H

        # Classe 0 (face)
        yolo_labels.append(f"0 {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

    # Sauvegarde fichier .txt
    with open(output_label_path, "w") as f_out:
        f_out.write("\n".join(yolo_labels))


In [4]:
import os
from pathlib import Path
from PIL import Image

INPUT_LABEL_FILE = "labels/val/wider_face_val_bbx_gt.txt"
INPUT_IMAGE_DIR = "images/val"
OUTPUT_IMAGE_DIR = "widerface_yolo/images/val"
OUTPUT_LABEL_DIR = "widerface_yolo/labels/val"

os.makedirs(OUTPUT_IMAGE_DIR, exist_ok=True)
os.makedirs(OUTPUT_LABEL_DIR, exist_ok=True)

with open(INPUT_LABEL_FILE, "r") as f:
    lines = f.readlines()

i = 0
while i < len(lines):
    image_rel_path = lines[i].strip()
    i += 1
    if i >= len(lines):
        break

    try:
        face_count = int(lines[i].strip())
    except ValueError:
        print(f"Erreur: ligne invalide pour le nombre de visages : {lines[i].strip()}")
        continue
    i += 1

    full_image_path = os.path.join(INPUT_IMAGE_DIR, image_rel_path)
    output_image_path = os.path.join(OUTPUT_IMAGE_DIR, os.path.basename(image_rel_path))
    output_label_path = os.path.join(OUTPUT_LABEL_DIR, Path(image_rel_path).with_suffix(".txt").name)

    # Copier l'image si elle existe
    if not os.path.exists(full_image_path):
        print(f"Image non trouvée : {full_image_path}")
        continue

    os.makedirs(os.path.dirname(output_image_path), exist_ok=True)
    os.system(f'cp "{full_image_path}" "{output_image_path}"')

    # Lire taille image
    try:
        with Image.open(full_image_path) as img:
            W, H = img.size
    except Exception as e:
        print(f"Erreur ouverture image {full_image_path} : {e}")
        continue

    yolo_labels = []
    for _ in range(face_count):
        if i >= len(lines):
            break
        parts = list(map(int, lines[i].strip().split()))
        i += 1
        if len(parts) < 4:
            continue  # ignorer lignes mal formées

        x, y, w, h = parts[0:4]

        # Ignorer les visages trop petits ou nuls
        if w <= 0 or h <= 0:
            continue

        # Conversion YOLO
        x_center = (x + w / 2) / W
        y_center = (y + h / 2) / H
        width = w / W
        height = h / H

        yolo_labels.append(f"0 {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

    # Sauvegarder les labels si au moins 1 visage valide
    if yolo_labels:
        with open(output_label_path, "w") as f_out:
            f_out.write("\n".join(yolo_labels))


In [6]:
from ultralytics import YOLO
model = YOLO("yolov8n.yaml")
model.train(data="data.yaml", imgsz=320, epochs=10, batch=4, device=0)


New https://pypi.org/project/ultralytics/8.3.149 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.148 🚀 Python-3.10.0 torch-2.7.0+cu126 CUDA:0 (Quadro T1000, 3897MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.yaml, momentum=0.937, mosaic=1.0, multi_scale=False, name=train23, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto

train: Scanning /home/addeche/Documents/Obscura/src/datasets/widerface_yolo/widerface_yolo/labels/train.cache... 1575 images, 2 backgrounds, 0 corrupt: 100%|██████████| 1576/1576 [00:00<?, ?it/s]

train: /home/addeche/Documents/Obscura/src/datasets/widerface_yolo/widerface_yolo/images/train/2_Demonstration_Protesters_2_231.jpg: 1 duplicate labels removed


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 314.9±97.6 MB/s, size: 99.9 KB)


val: Scanning /home/addeche/Documents/Obscura/src/datasets/widerface_yolo/widerface_yolo/labels/val.cache... 3222 images, 4 backgrounds, 0 corrupt: 100%|██████████| 3226/3226 [00:00<?, ?it/s]

...

      10/10     0.926G      2.914      1.941      1.243        202        320: 100%|██████████| 394/394 [00:26<00:00, 14.67it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 404/404 [00:21<00:00, 18.80it/s]


                   all       3226      39696      0.408      0.164      0.154     0.0606

10 epochs completed in 0.140 hours.
Optimizer stripped from /home/addeche/Documents/Obscura/runs/detect/train23/weights/last.pt, 6.2MB
Optimizer stripped from /home/addeche/Documents/Obscura/runs/detect/train23/weights/best.pt, 6.2MB

Validating /home/addeche/Documents/Obscura/runs/detect/train23/weights/best.pt...
Ultralytics 8.3.148 🚀 Python-3.10.0 torch-2.7.0+cu126 CUDA:0 (Quadro T1000, 3897MiB)
YOLOv8n summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 404/404 [00:18<00:00, 22.03it/s]


                   all       3226      39696      0.407      0.165      0.154     0.0607
Speed: 0.1ms preprocess, 2.0ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /home/addeche/Documents/Obscura/runs/detect/train23


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b36205b8160>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [10]:
best_model = YOLO("../../../runs/detect/train23/weights/best.pt")